# 09 - Ingestão Gold: Cálculo de KPIs e Persistência
**Squad 2 — Real Time for Business | Dupla 2**  
**Integrantes:** Kauan & Leandro  
**Tabelas de Escopo:** KPIs derivados de `silver_ecommerce_pedidos`  
**Branch:** `feat/squad2/kauan-leandro`  

### Objetivo da Task (Gold):
1. **Leitura da Silver:** Consumir `silver_ecommerce_pedidos` via Unity Catalog.
2. **Cálculo de KPIs:** Computar 3 KPIs de negócio usando PySpark window functions.
3. **Persistência Delta:** Overwrite nos KPIs correntes + Append nos históricos.
4. **Persistência SQL Server:** Overwrite via JDBC com sufixo `_dupla_2` (isolamento de squad).
5. **Controle de Segurança:** Flag `EXECUTAR_JDBC` para evitar escrita acidental no SQL Server compartilhado.

Notebook de cálculo e persistência de KPIs da camada Gold, lendo a Silver de pedidos.

## KPIs Calculados

| KPI | Tabela Gold (Delta) | Tabela SQL Server | Modo | Descrição |
| --- | --- | --- | --- | --- |
| Volume de Pedidos (5 min) | gold_kpi_volume_pedidos_5min | gold_kpi_volume_pedidos_5min_dupla_2 | Overwrite | Contagem e valor total por janela de 5 minutos |
| Ticket Médio por Pagamento | gold_kpi_ticket_medio_por_pagamento | gold_kpi_ticket_medio_por_pagamento_dupla_2 | Overwrite | Valor médio por método de pagamento |
| Taxa de Cancelamento (hora) | gold_kpi_taxa_cancelamento_hora | gold_kpi_taxa_cancelamento_hora_dupla_2 | Overwrite | % de cancelamentos por hora |

## Tabelas de Histórico (append)

| Tabela Delta | Modo | Descrição |
| --- | --- | --- |
| gold_hist_volume_pedidos_5min | Append | Histórico de volume por janela de 5 min |
| gold_hist_ticket_medio_por_pagamento | Append | Histórico de ticket médio por pagamento |
| gold_hist_taxa_cancelamento_hora | Append | Histórico de taxa de cancelamento por hora |

## Fluxo de Execução

1. **Setup** — Carrega credenciais e configura conexão JDBC + UC
2. **KPI 1** — Volume de pedidos em janelas de 5 minutos
3. **KPI 2** — Ticket médio por método de pagamento
4. **KPI 3** — Taxa de cancelamento por hora
5. **Persistência** — Overwrite nos KPIs (Delta + SQL Server) + Append nos históricos (Delta)
6. **Resumo** — Tabela consolidada com totais de cada KPI

**Arquitetura:** Leitura da Silver (Unity Catalog) → Cálculo de KPIs → Persistência em Delta (Gold) + SQL Server (JDBC)

In [0]:
%run ./00_setup_config

In [0]:
# Imports, configuração JDBC e funções auxiliares

from pyspark.sql import functions as F
import os

# --- Otimização Delta (alinhado com padrão Tech Lead) ---
spark.conf.set("spark.databricks.delta.properties.autoOptimize.optimizeWrite", "true")
spark.conf.set("spark.databricks.delta.properties.autoOptimize.autoCompact", "true")

# --- Configuração JDBC SQL Server ---
jdbc_url = f"jdbc:sqlserver://{os.getenv('SQL_HOST')}:1433;databaseName={os.getenv('SQL_DATABASE')};encrypt=true;trustServerCertificate=true;"
jdbc_props = {
    "user": os.getenv("SQL_USERNAME"),
    "password": os.getenv("SQL_PASSWORD"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# --- Funções auxiliares ---
def tabela_existe(nome_tabela):
    """Verifica se uma tabela existe no catálogo."""
    try:
        spark.table(nome_tabela).limit(1).count()
        return True
    except Exception:
        return False

def escrever_jdbc(df, tabela_sql, modo="overwrite"):
    """Escreve um DataFrame no SQL Server via JDBC com verificação de segurança."""
    total = df.count()
    if total == 0:
        print(f"  ⚠️ {tabela_sql}: DataFrame vazio — pulando escrita JDBC")
        return
    (df.write.format("sqlserver")
        .mode(modo)
        .option("url", jdbc_url)
        .option("dbtable", tabela_sql)
        .options(**jdbc_props)
        .save())
    print(f"  ✅ JDBC: {tabela_sql} — {total} registros gravados ({modo})")

def escrever_delta(df, tabela_uc, modo="overwrite"):
    """Escreve um DataFrame em tabela Delta no Unity Catalog."""
    total = df.count()
    if total == 0:
        print(f"  ⚠️ {tabela_uc}: DataFrame vazio — pulando escrita Delta")
        return
    (df.write.format("delta")
        .mode(modo)
        .option("optimizeWrite", "true")
        .option("autoCompact", "true")
        .saveAsTable(tabela_uc))
    print(f"  ✅ Delta: {tabela_uc} — {total} registros gravados ({modo})")

# --- Nomes das tabelas ---
TABELA_SILVER = "workspace.default.silver_ecommerce_pedidos"

# KPIs (overwrite)
TABELA_GOLD_KPI_1 = "workspace.default.gold_kpi_volume_pedidos_5min"
TABELA_GOLD_KPI_2 = "workspace.default.gold_kpi_ticket_medio_por_pagamento"
TABELA_GOLD_KPI_3 = "workspace.default.gold_kpi_taxa_cancelamento_hora"

# Histórico (append)
TABELA_GOLD_HIST_1 = "workspace.default.gold_hist_volume_pedidos_5min"
TABELA_GOLD_HIST_2 = "workspace.default.gold_hist_ticket_medio_por_pagamento"
TABELA_GOLD_HIST_3 = "workspace.default.gold_hist_taxa_cancelamento_hora"

print("Configuração concluída.")
print(f"JDBC URL: {jdbc_url[:50]}...")
print(f"Silver: {TABELA_SILVER}")

# --- Nomes das tabelas no SQL Server (com sufixo _dupla_2 para isolamento) ---
SQL_KPI_1 = "gold_kpi_volume_pedidos_5min_dupla_2"
SQL_KPI_2 = "gold_kpi_ticket_medio_por_pagamento_dupla_2"
SQL_KPI_3 = "gold_kpi_taxa_cancelamento_hora_dupla_2"

---
# 🟨 KPI 1 — Volume de Pedidos (janela de 5 minutos)

**Objetivo:** Contar pedidos e somar valor total em janelas de 5 minutos.

| Item | Valor |
| --- | --- |
| **Origem** | `workspace.default.silver_ecommerce_pedidos` |
| **Gold (KPI)** | `workspace.default.gold_kpi_volume_pedidos_5min` (overwrite) |
| **Gold (Histórico)** | `workspace.default.gold_hist_volume_pedidos_5min` (append) |
| **SQL Server** | `gold_kpi_volume_pedidos_5min` (overwrite) |
| **Métricas** | `qtd_pedidos`, `valor_total_pedidos`, `ticket_medio` |
| **Janela** | 5 minutos sobre `dt_pedido` |

In [0]:
# ============================================================================
# KPI 1: Volume de pedidos em janelas de 5 minutos
# ============================================================================

df_silver = spark.table(TABELA_SILVER)

# Cálculo do KPI: janela de 5 minutos sobre dt_pedido
df_kpi1 = (df_silver
    .filter(F.col("dt_pedido").isNotNull())
    .groupBy(F.window("dt_pedido", "5 minutes").alias("janela"))
    .agg(
        F.count("id_pedido").alias("qtd_pedidos"),
        F.sum("valor_total").alias("valor_total_pedidos"),
        F.avg("valor_total").alias("ticket_medio")
    )
    .select(
        F.col("janela.start").alias("dt_inicio_janela"),
        F.col("janela.end").alias("dt_fim_janela"),
        F.col("qtd_pedidos"),
        F.round("valor_total_pedidos", 2).alias("valor_total_pedidos"),
        F.round("ticket_medio", 2).alias("ticket_medio"),
        F.current_timestamp().alias("dt_calculo")
    )
    .orderBy("dt_inicio_janela"))

total_kpi1 = df_kpi1.count()
print(f"📊 KPI 1: {total_kpi1} janelas de 5 minutos calculadas")
display(df_kpi1.limit(10))

---
# 🟨 KPI 2 — Ticket Médio por Método de Pagamento

**Objetivo:** Calcular o ticket médio (valor_total) agrupado por método de pagamento.

| Item | Valor |
| --- | --- |
| **Origem** | `workspace.default.silver_ecommerce_pedidos` |
| **Gold (KPI)** | `workspace.default.gold_kpi_ticket_medio_por_pagamento` (overwrite) |
| **Gold (Histórico)** | `workspace.default.gold_hist_ticket_medio_por_pagamento` (append) |
| **SQL Server** | `gold_kpi_ticket_medio_por_pagamento` (overwrite) |
| **Métricas** | `ticket_medio`, `qtd_pedidos`, `valor_total` |
| **Normalização** | `metodo_pagamento` em upper + trim |

In [0]:
# ============================================================================
# KPI 2: Ticket médio por método de pagamento
# ============================================================================

# Normaliza metodo_pagamento (upper + trim) e agrega
df_kpi2 = (df_silver
    .filter(F.col("metodo_pagamento").isNotNull())
    .withColumn("metodo_pagamento_norm", F.upper(F.trim(F.col("metodo_pagamento"))))
    .groupBy("metodo_pagamento_norm")
    .agg(
        F.count("id_pedido").alias("qtd_pedidos"),
        F.sum("valor_total").alias("valor_total"),
        F.avg("valor_total").alias("ticket_medio")
    )
    .select(
        F.col("metodo_pagamento_norm").alias("metodo_pagamento"),
        F.col("qtd_pedidos"),
        F.round("valor_total", 2).alias("valor_total"),
        F.round("ticket_medio", 2).alias("ticket_medio"),
        F.current_timestamp().alias("dt_calculo")
    )
    .orderBy(F.col("valor_total").desc()))

total_kpi2 = df_kpi2.count()
print(f"📊 KPI 2: {total_kpi2} métodos de pagamento")
display(df_kpi2)

---
# 🟨 KPI 3 — Taxa de Cancelamento por Hora

**Objetivo:** Calcular a taxa de cancelamento (pedidos com `status_pedido = 'Cancelado'`) por hora.

| Item | Valor |
| --- | --- |
| **Origem** | `workspace.default.silver_ecommerce_pedidos` |
| **Gold (KPI)** | `workspace.default.gold_kpi_taxa_cancelamento_hora` (overwrite) |
| **Gold (Histórico)** | `workspace.default.gold_hist_taxa_cancelamento_hora` (append) |
| **SQL Server** | `gold_kpi_taxa_cancelamento_hora` (overwrite) |
| **Métricas** | `total_pedidos`, `pedidos_cancelados`, `taxa_cancelamento` |
| **Janela** | 1 hora sobre `dt_pedido` |

In [0]:
# ============================================================================
# KPI 3: Taxa de cancelamento por hora
# ============================================================================

# Janela de 1 hora: total de pedidos vs cancelados
df_kpi3 = (df_silver
    .filter(F.col("dt_pedido").isNotNull())
    .groupBy(F.window("dt_pedido", "1 hour").alias("janela"))
    .agg(
        F.count("id_pedido").alias("total_pedidos"),
        F.sum(F.when(F.col("status_pedido") == "Cancelado", 1).otherwise(0)).alias("pedidos_cancelados")
    )
    .select(
        F.col("janela.start").alias("dt_inicio_janela"),
        F.col("janela.end").alias("dt_fim_janela"),
        F.col("total_pedidos"),
        F.col("pedidos_cancelados"),
        F.round(F.col("pedidos_cancelados") / F.col("total_pedidos") * 100, 2).alias("taxa_cancelamento"),
        F.current_timestamp().alias("dt_calculo")
    )
    .orderBy("dt_inicio_janela"))

total_kpi3 = df_kpi3.count()
print(f"📊 KPI 3: {total_kpi3} janelas de 1 hora calculadas")
display(df_kpi3.limit(10))

---
# 💾 Persistência — Delta (Gold) + SQL Server

**Estratégia:**
* **KPIs (Delta)** — `overwrite` (estado atual, substituído a cada ciclo)
* **Histórico (Delta)** — `append` (acumula histórico a cada execução)
* **SQL Server (JDBC)** — `overwrite` (espelha o KPI atual, tabelas com sufixo `_dupla_2`)

As tabelas Gold são criadas no schema `workspace.default`.

In [0]:
# ============================================================================
# Persistência: Delta (overwrite KPIs + append histórico) + SQL Server (overwrite)
# ============================================================================

# Flag: alterar para True para habilitar escrita no SQL Server via JDBC
EXECUTAR_JDBC = False

print("=== Persistência Gold ===")
print(f"JDBC: {'ativado' if EXECUTAR_JDBC else 'desativado (apenas Delta)'}\n")

# --- KPI 1: Volume de pedidos 5 min ---
print("KPI 1 — Volume de pedidos (5 min):")
escrever_delta(df_kpi1, TABELA_GOLD_KPI_1, modo="overwrite")
escrever_delta(df_kpi1, TABELA_GOLD_HIST_1, modo="append")
if EXECUTAR_JDBC:
    escrever_jdbc(df_kpi1, SQL_KPI_1, modo="overwrite")

# --- KPI 2: Ticket médio por pagamento ---
print("\nKPI 2 — Ticket médio por pagamento:")
escrever_delta(df_kpi2, TABELA_GOLD_KPI_2, modo="overwrite")
escrever_delta(df_kpi2, TABELA_GOLD_HIST_2, modo="append")
if EXECUTAR_JDBC:
    escrever_jdbc(df_kpi2, SQL_KPI_2, modo="overwrite")

# --- KPI 3: Taxa de cancelamento por hora ---
print("\nKPI 3 — Taxa de cancelamento (hora):")
escrever_delta(df_kpi3, TABELA_GOLD_KPI_3, modo="overwrite")
escrever_delta(df_kpi3, TABELA_GOLD_HIST_3, modo="append")
if EXECUTAR_JDBC:
    escrever_jdbc(df_kpi3, SQL_KPI_3, modo="overwrite")

print("\n✅ Persistência concluída!")

---
# 📊 Resumo Geral

Consolida os resultados dos 3 KPIs Gold calculados, listando os totais em cada tabela Delta e confirmando a escrita no SQL Server.

In [0]:
# ============================================================================
# Resumo geral — KPIs Gold
# ============================================================================

print("=" * 60)
print("RESUMO GERAL — INGESTÃO GOLD KPIs")
print("=" * 60)

resumo_kpis = [
    ("KPI 1", "Volume pedidos (5min)", total_kpi1, TABELA_GOLD_KPI_1, TABELA_GOLD_HIST_1),
    ("KPI 2", "Ticket médio (pagamento)", total_kpi2, TABELA_GOLD_KPI_2, TABELA_GOLD_HIST_2),
    ("KPI 3", "Taxa cancelamento (hora)", total_kpi3, TABELA_GOLD_KPI_3, TABELA_GOLD_HIST_3),
]

print(f"{'KPI':<8} {'Descrição':<25} {'Registros':<12} {'Gold (Delta)':<45}")
print("-" * 90)
for kpi, desc, total, gold, hist in resumo_kpis:
    print(f"{kpi:<8} {desc:<25} {total:<12} {gold:<45}")

print()
print("Tabelas Gold (Delta — overwrite):")
for _, _, _, gold, _ in resumo_kpis:
    if tabela_existe(gold):
        n = spark.table(gold).count()
        print(f"  ✅ {gold}: {n} registros")
    else:
        print(f"  ⚠️ {gold}: não existe")

print()
print("Tabelas Histórico (Delta — append):")
for _, _, _, _, hist in resumo_kpis:
    if tabela_existe(hist):
        n = spark.table(hist).count()
        print(f"  ✅ {hist}: {n} registros")
    else:
        print(f"  ⚠️ {hist}: não existe")

print()
print("Tabelas SQL Server (JDBC — overwrite):")
for _, _, _, gold, _ in resumo_kpis:
    tabela_sql = gold.split(".")[-1]
    print(f"  ✅ {tabela_sql} (SQL Server)")